# SMART — generación sintética con IC-Light
Este notebook usa exclusivamente clips `train`, guarda el estado en Drive y construye dos ZIP autocontenidos: Raw+sintético (Mejora B) y LaMa+sintético (Mejora C). La validación oficial permanece fuera de estos ZIP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys, time, urllib.request
DRIVE_ROOT = Path('/content/drive/MyDrive/ia_article')
WORK = Path('/content/augmentation_work')
REPO = WORK / 'ia_article'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'chore/augmentation-evidence', '--filter=blob:none', '--sparse', 'https://github.com/unsa-semester-2026-A/ia_article.git', str(REPO)], check=True)
    subprocess.run(['git', 'sparse-checkout', 'set', 'experiments'], cwd=REPO, check=True)
os.chdir(REPO / 'experiments')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[cloud]'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', 'src/augmentation/test_pipeline.py', '-q'], check=True)


In [ ]:
# IC-Light is intentionally isolated: it reuses Colab CUDA Torch and never alters experiments/.
ICLIGHT = Path('/content/IC-Light')
subprocess.run([sys.executable, 'scripts/setup_iclight_colab.py', '--root', str(ICLIGHT)], check=True)
log_path = ICLIGHT / 'server_log.txt'
server = subprocess.Popen([str(ICLIGHT / '.venv/bin/python'), 'gradio_demo_bg.py'], cwd=ICLIGHT, stdout=log_path.open('w'), stderr=subprocess.STDOUT, env=os.environ | {'MPLBACKEND': 'agg'})
for _ in range(84):
    if server.poll() is not None: raise RuntimeError(log_path.read_text()[-4000:])
    try:
        urllib.request.urlopen('http://127.0.0.1:7860', timeout=2); break
    except OSError: time.sleep(5)
else: raise TimeoutError(log_path.read_text()[-4000:])
subprocess.run([sys.executable, '-m', 'src.augmentation.iclight', '--describe'], check=True)


In [ ]:
# Stage 1: select train-only tracks, RGBA crops and safe static slots.
DATA = DRIVE_ROOT / '04_augmentation'
RAW = DRIVE_ROOT / '01_processed/train_resized/train'
LAMA = DRIVE_ROOT / '03_lama_cleaning/result/train'
LABELS = DRIVE_ROOT / '01_processed/yolo_obb_labels/train'
subprocess.run([sys.executable, '-m', 'src.augmentation.run', 'prepare', '--split-metadata', str(DRIVE_ROOT/'01_processed/split_metadata.csv'), '--static-vehicles', str(DRIVE_ROOT/'02_pseudo_labeling/static_vehicles.json'), '--labels-train', str(LABELS), '--raw-images', str(RAW), '--lama-images', str(LAMA), '--workdir', str(DATA/'work')], check=True)
# Stage 2: IC-Light writes paired Raw/LaMa synthetic frames using exactly the same jobs.
subprocess.run([sys.executable, '-m', 'src.augmentation.run', 'render', '--jobs-jsonl', str(DATA/'work/jobs.jsonl'), '--output-dir', str(DATA/'work/synthetic')], check=True)
# Stage 3: package and sync two complete train datasets.
subprocess.run([sys.executable, '-m', 'src.augmentation.run', 'package', '--raw-images', str(RAW), '--lama-images', str(LAMA), '--base-labels', str(LABELS), '--synthetic-raw-images', str(DATA/'work/synthetic/raw/images'), '--synthetic-lama-images', str(DATA/'work/synthetic/lama/images'), '--synthetic-labels', str(DATA/'work/synthetic/raw/labels'), '--manifest', str(DATA/'work/synthetic/manifest.csv'), '--output-dir', str(DATA/'exports')], check=True)
